# Sistema de recomendação por similaridade de compra

Ranking de produtos com padrões de compra semelhantes ao **Motor de Popa 1949**.

## 1. Carregamento dos dados

In [1]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../data/1-lh_nautical_csv")

orders_df = (
    pd.read_csv(f"{DATA_PATH}/orders.csv", usecols=["id", "customer_id", "status"])
    .rename(columns={"id": "order_id"})
)

# Filtra por pedidos validos/validos (status "completed" ou "paid") e remove a coluna "status"
orders_df = orders_df[orders_df["status"].isin(["completed", "paid"])].drop(columns="status")

order_items_df = pd.read_csv(
    f"{DATA_PATH}/order_items.csv",
    usecols=["order_id", "product_variant_id"],
)

product_variants_df = (
    pd.read_csv(
        f"{DATA_PATH}/product_variants.csv",
        usecols=["id", "product_id"],
    )
    .rename(columns={"id": "product_variant_id"})
)

products_df = (
    pd.read_csv(f"{DATA_PATH}/products.csv", usecols=["id", "name"])
    .rename(columns={"id": "product_id"})
)

print(f"orders: linhas {orders_df.shape[0]} colunas {orders_df.shape[1]}")
display(orders_df.head(1))
print(f"order_items: linhas {order_items_df.shape[0]} colunas {order_items_df.shape[1]}")
display(order_items_df.head(1))
print(f"product_variants: linhas {product_variants_df.shape[0]} colunas {product_variants_df.shape[1]}")
display(product_variants_df.head(1))
print(f"products: linhas {products_df.shape[0]} colunas {products_df.shape[1]}")
display(products_df.head(1))

orders: linhas 34365 colunas 2


,order_id,customer_id
0,1,1136


order_items: linhas 147320 colunas 2


,order_id,product_variant_id
0,1,113


product_variants: linhas 1009 colunas 2


,product_variant_id,product_id
0,1,1


products: linhas 500 colunas 2


,product_id,name
0,1,Motor de Popa 6014


## 2. Construção da matriz usuário–produto

In [2]:
customer_product_df = (
    orders_df
    .merge(order_items_df, on="order_id", how="inner")
    .merge(product_variants_df, on="product_variant_id", how="inner")
    [["customer_id", "product_id"]]
)

# Um usuário pode realizar multiplas orders de um mesmo produto
print(f"Linhas antes de remover repetições: {len(customer_product_df):,}")

customer_product_df = (
    customer_product_df
    .drop_duplicates()
    .assign(interaction=1)
)

print(f"Interações únicas: {len(customer_product_df):,}")
display(customer_product_df.head())

Linhas antes de remover repetições: 103,291
Interações únicas: 97,358


,customer_id,product_id,interaction
0,1136,59,1
1,618,146,1
2,618,180,1
3,618,275,1
4,618,190,1


In [3]:
# Construção da matriz usuário–produto, caso um usuário não tenha interações com um produto, o valor será 0
user_item_matrix = (
    customer_product_df
    .pivot(index="customer_id", columns="product_id", values="interaction")
    .fillna(0)
    .astype(int)
)

print(f"Dimensões da matriz usuário–produto: linhas {user_item_matrix.shape[0]} X colunas {user_item_matrix.shape[1]}")
print(f"Valores presentes na matriz: {sorted(user_item_matrix.stack().unique().tolist())}")
display(user_item_matrix)

Dimensões da matriz usuário–produto: linhas 2000 X colunas 500
Valores presentes na matriz: [0, 1]


product_id,1,2,3,4,5,6,7,8,9,10,...,491,492,493,494,495,496,497,498,499,500
customer_id,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,1,0,...,0,0,1,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,1,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,0
5,0,0,1,0,0,1,0,0,1,0,...,0,1,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1996,0,0,0,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1998,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,0


## 3. Similaridade entre produtos

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

# Construção da matriz produto X produto (coluna X coluna)
product_similarity_df = pd.DataFrame(
    cosine_similarity(user_item_matrix.T),
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns,
)

print(f"Dimensões da matriz produto–produto: Linhas {product_similarity_df.shape[0]} X Colunas {product_similarity_df.shape[1]}")
display(product_similarity_df)

Dimensões da matriz produto–produto: Linhas 500 X Colunas 500


product_id,1,2,3,4,5,6,7,8,9,10,...,491,492,493,494,495,496,497,498,499,500
product_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.160902,0.125539,0.121489,0.175845,0.135929,0.106406,0.094334,0.134731,0.095238,...,0.123091,0.100235,0.153083,0.095831,0.069985,0.124656,0.143156,0.146064,0.074705,0.106056
2,0.160902,1.000000,0.137170,0.124936,0.184451,0.139786,0.089162,0.109945,0.142107,0.079577,...,0.090417,0.098784,0.161004,0.093076,0.071971,0.089735,0.147217,0.153872,0.094554,0.100340
3,0.125539,0.137170,1.000000,0.093416,0.133860,0.136680,0.150002,0.072536,0.147429,0.054924,...,0.094649,0.072257,0.140449,0.061406,0.073994,0.115022,0.153033,0.152033,0.066280,0.112538
4,0.121489,0.124936,0.093416,1.000000,0.116311,0.064550,0.114695,0.016639,0.100542,0.062994,...,0.069786,0.110499,0.105857,0.084515,0.054006,0.104440,0.129331,0.103681,0.068418,0.106627
5,0.175845,0.184451,0.133860,0.116311,1.000000,0.129735,0.101871,0.123549,0.157168,0.067700,...,0.084848,0.103640,0.165477,0.104592,0.078393,0.128891,0.155239,0.184177,0.106952,0.135977
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,0.124656,0.089735,0.115022,0.104440,0.128891,0.110703,0.072216,0.061462,0.088627,0.050901,...,0.050124,0.107144,0.144508,0.097559,0.078372,1.000000,0.106636,0.134915,0.126363,0.134744
497,0.143156,0.147217,0.153033,0.129331,0.155239,0.135957,0.097093,0.090381,0.134759,0.097765,...,0.090255,0.111469,0.139287,0.103839,0.047895,0.106636,1.000000,0.120683,0.088485,0.143708
498,0.146064,0.153872,0.152033,0.103681,0.184177,0.167923,0.132087,0.065869,0.141116,0.087282,...,0.085949,0.118108,0.145760,0.111525,0.073302,0.134915,0.120683,1.000000,0.060189,0.124411


## 4. Ranking para "Motor de Popa 1949"

In [5]:
TARGET_PRODUCT = "Motor de Popa 1949"

target_product_id = products_df.loc[
    products_df["name"].eq(TARGET_PRODUCT),
    "product_id",
].iloc[0]

print(f"ID do produto alvo: {target_product_id}")

ranking_df = (
    product_similarity_df.loc[target_product_id]
    .drop(index=target_product_id) # Removendo o próprio produto da lista de recomendações
    .rename("similarity")
    .sort_values(ascending=False)
    .head(5)
    .rename_axis("product_id")
    .reset_index()
    .merge(products_df, on="product_id", how="left")
    [["name", "similarity"]]
)

display(ranking_df)

ID do produto alvo: 180


,name,similarity
0,Vela Mestra 1913,0.204586
1,Cabo Náutico 2105,0.189809
2,Motor de Popa 6014,0.187209
3,asdf,0.185391
4,Âncora Bruce 7665,0.185325


## 5. Respostas

### 7.2 Qual é o nome do produto com MAIOR similaridade ao “Motor de Popa 1949”?

In [6]:
best_product_name = ranking_df.iloc[0]["name"]
print(
    f'O produto com maior similaridade ao "{TARGET_PRODUCT}" é "{best_product_name}".'
)

O produto com maior similaridade ao "Motor de Popa 1949" é "Vela Mestra 1913".


### 7.3 Explique

1. **Como a matriz foi construída?** 

Cada linha representa um cliente e cada coluna representa um produto. O valor `1` indica que o cliente comprou o produto pelo menos uma vez, enquanto `0` indica ausência de compra. Quantidades e compras repetidas foram ignoradas.

2. **O que significa a similaridade de cosseno nesse contexto?**

 Ela mede o quanto dois produtos possuem padrões semelhantes de compradores. Quanto mais clientes em comum compraram os dois produtos, maior tende a ser a similaridade entre seus vetores.

3. **Limitação:** 

o método ignora quando as compras ocorreram. Assim, compras realizadas em momentos muito distantes contribuem da mesma forma que compras feitas em períodos próximos.